# Week 2: Encoding, Hashing, and File Integrity
**Goal:** tell encoding, hashing, and encryption apart, then build a tool that catches a hacked website file.

**Time:** about 2 hours. **You need:** Week 1 skills (files, loops, dictionaries, functions).

**Story.** The Riverside Library website was hacked last month. The IT team saved a list of the SHA-256 hash of
every website file when the site was known to be clean (`baseline_hashes.json`). Today the site looks normal.
Your job: *has any file been changed, and what did the attacker add?*

> All files are synthetic. The "attacker" domain `example.net` is reserved and reaches nothing.

## Part 0: Get the data (run this cell)

In [ ]:
import os, urllib.request

DATA_URL = "https://raw.githubusercontent.com/Pantea1358/Cyber-labs-/main/weeks/week02-hashing-integrity/data/"
FILES = ["baseline_hashes.json", "website/index.html", "website/about.html",
         "website/login.html", "website/login.js", "website/style.css"]
for name in FILES:
    path = os.path.join("data", name)
    if not os.path.exists(path):
        os.makedirs(os.path.dirname(path), exist_ok=True)
        urllib.request.urlretrieve(DATA_URL + name, path)
print("Data ready:", sorted(os.listdir("data/website")))

## Part 1: Encoding is not security (15 min)
**Encoding** changes how data looks so computers can move it around. Anyone can reverse it. No key needed.
Base64 is a common encoding.

1. Encode the text `"my password is tiger42"` with Base64 and print it.
2. A website stored this "hidden" admin password: `c3VwZXJzZWNyZXQxMjM=`. Decode it.

*Hint:* `base64.b64encode(text.encode())` and `base64.b64decode(data).decode()`

In [ ]:
import base64

# TODO: encode "my password is tiger42" and print it
encoded = None
print("Encoded:", encoded)

hidden = "c3VwZXJzZWNyZXQxMjM="
# TODO: decode `hidden` and print it

**Question 1:** you decoded the password with no key. So is Base64 a safe way to hide passwords? Why do people still
mistake it for encryption?

*Your answer:*

## Part 2: Hashing, a fingerprint for data (15 min)
A **hash function** turns any input into a fixed-length fingerprint. SHA-256 always gives 64 hex characters.

- Same input, same hash, every time.
- A tiny change gives a totally different hash (the *avalanche effect*).
- You cannot turn the hash back into the input.

Write `sha256_text(text)` that returns the SHA-256 hex digest of a string. Then compare `"hello"` and `"Hello"`:
how many of the 64 characters are different?

In [ ]:
import hashlib

def sha256_text(text):
    # TODO: return the SHA-256 hex digest of text
    return ""

a = sha256_text("hello")
b = sha256_text("Hello")
print(a)
print(b)
# TODO: count how many characters differ between a and b

In [ ]:
assert sha256_text("hello") == "2cf24dba5fb0a30e26e83b2ac5b9e29e1b161e5c1fa7425e73043362938b9824"
print("sha256_text works!")

## Part 3: Checking a password without storing it (10 min)
Websites should store the **hash** of a password, not the password. At login they hash what you typed and compare.

The stored hash for user `maya` is below. Which of the guesses is her password?

In [ ]:
stored = "faf3541b3be3647699a6588ee15805456bf99acd1cb2edc05992307e1aeb694e"
guesses = ["password", "maya2008", "letmein", "riverside", "qwerty"]

# TODO: hash each guess and print the one that matches

**Question 2:** you just "cracked" a password by guessing. Next week we will do this at scale.
What does that tell you about storing short, common passwords, even when they are hashed?

*Your answer:*

## Part 4: Hash a file (10 min)
Files are bytes, not text. Write `sha256_file(path)` that opens the file in binary mode (`"rb"`) and returns its SHA-256.

*Hint:* `hashlib.sha256(f.read()).hexdigest()` works for small files. For big files you would read in chunks.

In [ ]:
def sha256_file(path):
    # TODO: open the file in "rb" mode and return its SHA-256 hex digest
    return ""

print(sha256_file("data/website/index.html"))

## Part 5: Build the integrity checker (20 min)
Load `data/baseline_hashes.json` (a dictionary of filename to hash). For every file, compare today's hash with the baseline and print one of:

- `OK       filename`
- `CHANGED  filename`
- `MISSING  filename` (in the baseline but gone now)
- `NEW      filename` (on disk but not in the baseline)

*Hint:* `json.load(f)` and `os.listdir("data/website")`

In [ ]:
import json

# TODO: load the baseline dictionary
baseline = {}

# TODO: build a dictionary of today's hashes for every file in data/website
current = {}

changed = []
# TODO: compare and print OK / CHANGED / MISSING / NEW for every file
#       add changed filenames to the `changed` list

## Part 6: Investigate (10 min)
Print the changed file. Find the line the attacker added.

**Question 3:** in plain words, what does the attacker's line do? Why is it hard to notice just by visiting the website?

*Your answer:*

In [ ]:
# TODO: print the contents of each file in `changed`

## Part 7: AI twist, fact-check the AI (30 min)
AI assistants sound confident even when they are wrong or out of date. Ask an AI assistant (if your school allows it)
these questions and fact-check each answer using what you learned today and a trusted source
(for example [OWASP](https://owasp.org) or [NIST](https://www.nist.gov)).

| # | Question to ask the AI | AI's answer (short) | Correct? | Your evidence |
| --- | --- | --- | --- | --- |
| 1 | Is Base64 a form of encryption? | | | |
| 2 | Is MD5 safe for checking that a file was not tampered with by an attacker? | | | |
| 3 | Is plain SHA-256 a good way to store user passwords? | | | |
| 4 | Can you reverse a SHA-256 hash to get the original text? | | | |

Then run the cell below to compare hash lengths, and look up why MD5 is called "broken".

In [ ]:
for name in ["md5", "sha1", "sha256", "sha512"]:
    digest = hashlib.new(name, b"hello").hexdigest()
    print(f"{name:<7} {len(digest) * 4:>4} bits  {digest[:32]}...")

## Part 8: Reflect (15 min)
**Exit ticket.** Answer in 2 to 3 sentences each:

1. Explain the difference between encoding, hashing, and encryption to a friend who does not study computers.
2. Your integrity checker needs the baseline file to be trustworthy. Where should the IT team keep it, and why?

*Your answers:*

**Stretch (optional):** make your checker also print *which lines* changed. Look up Python's `difflib`.